# Middleware

Middleware provides a way to more tightly control what happens inside the agent.

Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging
- Transforming prompts, tool selection, and output formatting
- Adding retries, fallbacks, and early termination logic
- Applying rate limits, guardrails, and PII detection

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

## Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context.

Summarization is useful for the following:

- Long-running conversations that exceed context windows
- Multi-turn dialogues with extensive history
- Applications where preserving full conversation context matters

In [4]:
from langchain_core.messages import (
    HumanMessage,
    SystemMessage
)

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI



# OpenRouter model
llm = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    base_url="https://openrouter.ai/api/v1",
)


# Message-based summarization
agent = create_agent(
    model=llm,

    checkpointer=InMemorySaver(),

    middleware=[
        SummarizationMiddleware(
            model=llm,

            trigger=("messages", 20),

            keep=("messages", 4)
        )
    ]
)


# Run with thread ID
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}


# Example conversation
questions = [
    "Hello!",
    "My name is John.",
    "I am learning LangChain.",
    "Can you summarize our conversation?"
]


# Invoke agent repeatedly
for question in questions:

    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=question)
            ]
        },
        config=config
    )

    print(response["messages"][-1].content)

Hello! 👋 How can I help you today?
Nice to meet you, John! How can I assist you today?
That’s awesome, John! LangChain is a powerful framework for building applications that integrate large language models (LLMs) with other tools and data sources.

What’s your current focus? Are you:

1. 👟 just starting out and looking for a quick‑start tutorial?  
2. 🛠️ working on a specific project (e.g., a chatbot, retrieval‑augmented generation, or data‑analysis tool)?  
3. 📚 exploring concepts like chains, agents, memory, or evaluators?  
4. 🔧 troubleshooting an error or debugging a notebook?

Let me know what you’d like to dive into next—and I can share code snippets, explanations, or resources to help you move forward!
Sure thing! Here’s a quick recap of what we’ve covered so far:

1. **Introduction** – You greeted me, and I introduced myself.  
2. **Name** – You told me your name is John.  
3. **Learning LangChain** – You mentioned you’re learning LangChain.  
4. **Request for Summary** – You a

## Human-in-the-Loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute.

Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions)
- Compliance workflows where human oversight is mandatory
- Long-running conversations where human feedback guides the agent

In [6]:
## Step 1 — Create a Tool

from langchain.tools import tool

@tool
def send_email(to: str, subject: str):
    """Send an email."""

    return f"Email sent to {to} with subject '{subject}'"

In [8]:
#  Create Middleware

from langchain.agents.middleware import HumanInTheLoopMiddleware

human_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": True
    }
)


''' Whenever send_email tool is called:
PAUSE execution'''

' Whenever send_email tool is called:\nPAUSE execution'

In [20]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import (
    InMemorySaver
)


model = ChatGroq(
    model="qwen/qwen3-32b"
)

agent = create_agent(
    model=model,
    tools=[send_email],
    checkpointer=InMemorySaver(),
    middleware=[human_middleware]
)

In [21]:
from langgraph.types import Command


# Thread configuration
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}


# Step 1: Run agent
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Send an email to john@example.com "
                    "with subject Meeting Tomorrow"
                )
            }
        ]
    },
    config=config
)

#print(response)
print("/n/n")

# Step 2: Approve interrupted tool call
if "__interrupt__'" in response:

    print("Paused! 🫸🫸🫸🫸 Approving...")


    # Resume execution
    response = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "approve"
                    }
                ]
            }
        ),
        config=config
    )

    print(
        f"Result: {response['messages'][-1].content}"
    )

/n/n


In [23]:
from langchain.tools import tool

from langchain.agents import create_agent

from langchain.agents.middleware import (
    HumanInTheLoopMiddleware
)

from langchain_groq import ChatGroq

from langgraph.checkpoint.memory import (
    InMemorySaver
)

from langgraph.types import Command


# Tool
@tool
def send_email(to: str, subject: str):
    """Send an email."""

    return f"Email sent to {to} with subject '{subject}'"


# Model
model = ChatGroq(
    model="qwen/qwen3-32b"
)


# Middleware
human_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": True
    }
)


# Agent
agent = create_agent(
    model=model,

    tools=[send_email],

    middleware=[human_middleware],

    checkpointer=InMemorySaver()
)


# Config
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}


# Step 1: Invoke agent
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Send an email to john@example.com "
                    "with subject Meeting Tomorrow"
                )
            }
        ]
    },
    config=config
)


# Print interruption response
print("FIRST RESPONSE:")
print(response)


# Step 2: Resume if interrupted
if "__interrupt__" in response:

    print("\nPaused for approval...\n")

    response = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "approve"
                    }
                ]
            }
        ),
        config=config
    )

    print("\nFINAL RESPONSE:")
    print(response)

    print("\nAssistant Output:")
    print(response["messages"][-1].content)

else:
    print("\nNo interruption happened.")

FIRST RESPONSE:
{'messages': [HumanMessage(content='Send an email to john@example.com with subject Meeting Tomorrow', additional_kwargs={}, response_metadata={}, id='ef2d29c0-95f1-4ccf-be74-47a2c8aab4f0'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to john@example.com with the subject "Meeting Tomorrow". Let me check the available tools. There\'s a function called send_email that requires \'to\' and \'subject\' parameters. The \'to\' address is provided as john@example.com, and the subject is "Meeting Tomorrow". I need to make sure both parameters are included in the function call. The body of the email isn\'t specified, but the function doesn\'t require it, so I can omit it. Just need to structure the JSON correctly with the required fields. Let me double-check the parameters\' data types. Both are strings, so that\'s correct. Alright, the tool call should look like this.\n', 'tool_calls': [{'id': 'a7wtwqhfr', 'function': {'arg

## Reject

In [ ]:
from langgraph.types import Command


# Reject tool execution
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject"
                }
            ]
        }
    ),
    config=config
)

print(response)

## Edit

In [24]:
from langgraph.types import Command


# Edit tool arguments
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",

                    "edited_action": {
                        "tool": "send_email",

                        "args": {
                            "to": "newperson@example.com",

                            "subject": "Updated Meeting Subject"
                        }
                    }
                }
            ]
        }
    ),
    config=config
)

print(response)

{'messages': [HumanMessage(content='Send an email to john@example.com with subject Meeting Tomorrow', additional_kwargs={}, response_metadata={}, id='ef2d29c0-95f1-4ccf-be74-47a2c8aab4f0'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to john@example.com with the subject "Meeting Tomorrow". Let me check the available tools. There\'s a function called send_email that requires \'to\' and \'subject\' parameters. The \'to\' address is provided as john@example.com, and the subject is "Meeting Tomorrow". I need to make sure both parameters are included in the function call. The body of the email isn\'t specified, but the function doesn\'t require it, so I can omit it. Just need to structure the JSON correctly with the required fields. Let me double-check the parameters\' data types. Both are strings, so that\'s correct. Alright, the tool call should look like this.\n', 'tool_calls': [{'id': 'a7wtwqhfr', 'function': {'arguments': '{"subj